# Scenario 2 — Zero-Shot: No Labelled Training Data
## Decoder Notebook (Qwen2.5-1.5B-Instruct)

**Conditions:** Identical to `zero-bert-robert_E-F.ipynb`  
- Same dataset: `puyang2025/seven-phishing-email-datasets`  
- Same 7 corpora, same `sample(500, random_state=42)` per corpus  
- Same `evaluate()` signature & final summary table format  
- NO fine-tuning — zero-shot prompt only  

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` loaded in 4-bit (bitsandbytes) to fit on a single T4  

**Expected:** 70–85% accuracy across corpora vs encoder ~50% coin-flip baseline  
*(Literature ref: arXiv 2505.00034 — Qwen2.5-1.5B zero-shot 77.1% on SpamAssassin)*

In [ ]:
# ── Cell 1: Environment setup (mirrors encoder notebook exactly) ──────────────
!nvidia-smi
!pip install "numpy<2.0" -q
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall -q
!pip install transformers accelerate bitsandbytes datasets scikit-learn pandas tqdm -q

In [1]:
# ── Cell 2: Imports & config ──────────────────────────────────────────────────
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_score, recall_score)
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                           BitsAndBytesConfig)
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Identical evaluate() to encoder notebook ──────────────────────────────────
def evaluate(y_true, y_pred, name="", ms=None):
    r = {
        "Model":     name,
        "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}",
    }
    if ms:
        r["ms/sample"] = f"{ms:.2f}"
    return r

# ── Model config ──────────────────────────────────────────────────────────────
DECODER_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantisation: keeps peak VRAM under 6 GB on a T4
BNB_CFG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Decoder model : {DECODER_MODEL_ID}")
print("Quantisation  : 4-bit NF4 (bitsandbytes)")

Device: cuda
GPU: Tesla T4
Decoder model : Qwen/Qwen2.5-1.5B-Instruct
Quantisation  : 4-bit NF4 (bitsandbytes)


In [3]:
# ── Cell 3: Load dataset — identical to encoder notebook ─────────────────────
print("Loading puyang2025/seven-phishing-email-datasets...")
ds_p   = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_puy = ds_p.to_pandas()

# Inspect actual columns — dataset schema may differ from docs
print(f"Columns: {df_puy.columns.tolist()}")

# Build 'text' robustly from whatever text columns are present
# Priority: subject + body → subject + text → text → email → first str column
def build_text_column(df):
    cols = set(df.columns)
    if "subject" in cols and "body" in cols:
        return (df["subject"].fillna("") + " " + df["body"].fillna("")).str.strip()
    if "subject" in cols and "text" in cols:
        return (df["subject"].fillna("") + " " + df["text"].fillna("")).str.strip()
    for candidate in ("text", "email", "content", "message"):
        if candidate in cols:
            return df[candidate].fillna("").str.strip()
    # Last resort: first object/string column that is not label or dataset_name
    for col in df.columns:
        if col not in ("label", "dataset_name") and df[col].dtype == object:
            print(f"  → Falling back to column: '{col}'")
            return df[col].fillna("").str.strip()
    raise ValueError(f"No usable text column found. Columns: {df.columns.tolist()}")

df_puy["text"] = build_text_column(df_puy)

df_puy = (
    df_puy[["text", "label", "dataset_name"]]
    .drop_duplicates("text")
    .dropna()
    .reset_index(drop=True)
)
df_puy["label"] = df_puy["label"].astype(int)
print(f"Total: {len(df_puy):,}")
print(df_puy["dataset_name"].value_counts().to_string())

Loading puyang2025/seven-phishing-email-datasets...
Columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
Total: 162,261
dataset_name
TREC-05     44393
TREC-07     42913
CEAS-08     31144
Enron       23834
TREC-06     13102
Assassin     4584
Ling         2291


In [4]:
# ── Cell 4: Zero-shot prompt & inference helper ───────────────────────────────
#
# Prompt design notes:
#   • Role framing  : "cybersecurity email analyst" grounds the task
#   • Constrained output: "Reply with only ONE word" minimises parse failures
#   • Label mapping : PHISHING → 1,  LEGITIMATE → 0  (same as dataset labels)
#   • Email truncated to 512 chars to stay within a T4's context budget;
#     matches the truncation used in Scenario_2_Decoder_ZeroShot.ipynb

PHISH_PROMPT = """You are a cybersecurity email analyst.
Classify the following email as either PHISHING or LEGITIMATE.
Reply with only ONE word: PHISHING or LEGITIMATE.

Email:
{email}

Classification:"""


def llm_zero_shot(texts, model, tokenizer, desc="Qwen zero-shot"):
    """Run zero-shot inference on a list of email strings.
    Returns a list of int labels: 1 = PHISHING, 0 = LEGITIMATE.
    """
    preds = []
    for text in tqdm(texts, desc=desc):
        prompt  = PHISH_PROMPT.format(email=text[:512])
        inputs  = tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=5,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False,          # greedy — deterministic output
            )
        # Decode only the newly generated tokens
        decoded = tokenizer.decode(
            out[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()
        preds.append(1 if "PHISH" in decoded else 0)
    return preds


print("Prompt template ready.")
print("Label mapping  : PHISHING → 1 | LEGITIMATE → 0")

Prompt template ready.
Label mapping  : PHISHING → 1 | LEGITIMATE → 0


In [7]:
# ── Cell 5: Load model (once — reused across all corpora) ────────────────────
#
# Loading once and iterating over corpora is much faster than the encoder
# approach (which reloads per model × corpus) because the LLM is large.
# Memory is freed at the end of Cell 6.

print(f"Loading {DECODER_MODEL_ID} in 4-bit ...")
tok = AutoTokenizer.from_pretrained(
    DECODER_MODEL_ID, trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL_ID,
    quantization_config=BNB_CFG,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

# Show approximate GPU memory footprint
if DEVICE == "cuda":
    used_mb = torch.cuda.memory_allocated() / 1e6
    print(f"GPU memory allocated: {used_mb:.0f} MB")

Loading Qwen/Qwen2.5-1.5B-Instruct in 4-bit ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded.
GPU memory allocated: 1118 MB


In [ ]:
# ── Cell 6: Run over all 7 corpora — identical sample logic to encoder ────────
#
# Encoder notebook:
#   df_test = df_puy[df_puy["dataset_name"] == corpus]
#             .sample(min(500, len(...)), random_state=42)
# We replicate this exactly.

CORPORA = df_puy["dataset_name"].unique().tolist()
all_results = []

for corpus in CORPORA:
    df_corpus = df_puy[df_puy["dataset_name"] == corpus]
    df_test = df_puy[df_puy["dataset_name"] == corpus].reset_index(drop=True)
    X_zs = df_test["text"].values
    y_zs = df_test["label"].values

    legit_n = (df_test.label == 0).sum()
    spam_n  = (df_test.label == 1).sum()
    print(f"\n{'='*30}")
    print(f"DATASET: {corpus}")
    print(f"{'='*30}")
    print(f"Test set: {len(df_test):,} | Legit: {legit_n} | Spam: {spam_n}")

    t0    = time.time()
    preds = llm_zero_shot(X_zs, model, tok, desc=f"Qwen on {corpus}")
    ms    = (time.time() - t0) / len(X_zs) * 1000

    row = evaluate(y_zs, preds, "Qwen2.5-1.5B (zero-shot)", ms=ms)
    row["Dataset"] = corpus
    all_results.append(row)

# Free VRAM before printing results
del model
torch.cuda.empty_cache()
print("\nModel unloaded from GPU.")


DATASET: TREC-07
Test set: 42,913 | Legit: 19429 | Spam: 23484


Qwen on TREC-07:   0%|          | 0/42913 [00:00<?, ?it/s]

In [ ]:
# ── Cell 7: Final summary table — same format as encoder notebook ─────────────
df_results = pd.DataFrame(all_results)[
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1", "ms/sample"]
]

print("\n" + "="*60)
print("SCENARIO 2 — DECODER (ZERO-SHOT) RESULTS — ALL CORPORA")
print("="*60)
print(df_results.to_string(index=False))

print("\n→ Compare against encoder notebook (zero-bert-robert_E-F.ipynb):")
print("  Encoders with random heads: ~40-80% accuracy but F1 near 0 on")
print("  minority class — essentially predicting the majority class.")
print("  Qwen2.5-1.5B zero-shot should show meaningful F1 across corpora")
print("  with NO training data required.")

In [ ]:
# ── Cell 8 (optional): Per-corpus breakdown with classification report ─────────

print("Notebook complete.")